# async/await, asyncio.gather, concurrency for API calls

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### async/await

> **Problem.** A web service answers questions with the model. Each answer takes about a second. Written as ordinary functions, the service handles one user at a time: with 50 people asking at once, the 50th waits almost a minute — and the CPU was idle the whole time, because all it did was wait for the network.

**Idea.** A function that can pause while it waits, so one program serves many requests at the same time.

**Use when** the work is waiting on a network or database (API calls, web handlers, workers).  
**Not when** the work is heavy calculation — it never pauses, so it blocks everyone.

```
normal    send ──▶ frozen for 1 s ──────────────▶ answer
async     send ──▶ await (paused, others run) ──▶ answer
                         │
                         └─ user 2's request runs here
```

**How it works.**
1. `async def ask(city)` declares a function that is allowed to pause.
2. Inside it, `await ai.chat.completions.create(...)` sends the request and pauses *this* function. The event loop is free.
3. While it waits, the loop runs whatever else is ready — another user's `ask`, a database call.
4. When the reply arrives, the loop resumes `ask` exactly where it paused and it returns the text.
5. `run_async(...)` (or `asyncio.run(...)` in a script) starts the loop once and runs the whole thing.

| | what happens | result |
|:--|:--|:--|
| ✗ normal | 3 users → one after another | 3 s total |
| ✓ async | 3 users → all waiting together | ≈1 s total |
| ✗ forgot await | call without `await` | a coroutine object instead of an answer |

**Production code and its real output**

In [2]:
# async/await — one API call on the async client. `await` frees the event loop during the
# network wait; FastAPI handlers and worker services run on this pattern.


async def ask_country(city: str) -> str:
    async with async_client() as ai:
        response = await ai.chat.completions.create(
            model=settings.openai_model,
            messages=[{"role": "user", "content": f"Which country is {city} in? One word."}],
            temperature=0,
        )
    return response.choices[0].message.content


answer = run_async(ask_country("Chennai"))
print("answer:", answer)
assert "India" in answer

answer: India.


**What the output shows.** One call, one answer (`India.`). The speed-up only appears when several requests overlap — next item.

**In practice**
- **CPU work** — tokenising, local embeddings or big loops inside `async def` never pause, so they freeze every other request; run them in a process pool.
- **one loop** — start the loop once at the entry point; a client is bound to the loop it was created in, and reusing it in another loop fails with "Event loop is closed".
- **contagion** — an async function can only be awaited from another async function, so async spreads upward to the entry point — plan for it from the start.
- **missing await** — forgetting `await` returns a coroutine object instead of a result and nothing runs; pyright flags it, so keep the type checker on.

**Alternatives** — threads (heavier, simpler mental model, fine up to a few hundred) · a process pool for CPU-bound work

**Terms** — *event loop*: the scheduler that switches between paused functions · *coroutine*: an async function that has started but not finished · *await*: the pause point


### asyncio.gather

> **Problem.** Your indexing job embeds 1,000 documents. Each call to the model takes 0.6 s. Written as a normal loop, the job takes 10 minutes, and the CPU is idle for 9 minutes and 59 seconds of it — the program is just waiting for the network, one reply at a time.

**Idea.** Start all the requests together and wait once for all of them.

**Use when** the requests do not depend on each other (embed 1,000 chunks, grade 50 answers, call three providers).  
**Not when** step 2 needs step 1's answer — then there is nothing to run in parallel.

```
one after another   q1 ──▶ q2 ──▶ q3 ──▶ q4 ──▶ q5          3.0 s

gather              q1 ──┐
                    q2 ──┤
                    q3 ──┼── all in flight ──▶ done        0.6 s
                    q4 ──┤
                    q5 ──┘
```

**How it works.**
1. `ask(ai, city)` is an async function: calling it does *not* run it yet — it returns a coroutine, a request prepared but not sent.
2. You build five of these in a list. Still nothing has gone over the network.
3. `await asyncio.gather(*tasks)` hands all five to the event loop at once; five requests leave in the same millisecond.
4. As each reply arrives, its coroutine finishes. `gather` waits for the last one.
5. It returns one list, in the same order as the input — regardless of which reply came back first.

| | what happens | result |
|:--|:--|:--|
| ✗ loop | 5 × 0.6 s, one at a time | 3.0 s |
| ✓ gather | 5 at once, wait once | 0.6 s, same answers, same order |
| ✗ one fails | default: whole gather raises | use `return_exceptions=True` to keep the rest |

**Production code and its real output**

In [3]:
# asyncio.gather — fan out independent calls and wait for all. This is the single biggest
# latency win in batch jobs and RAG pipelines.
import time

CITIES = ["Chennai", "Paris", "Oslo", "Tokyo", "Lima"]


async def ask_country(ai: AsyncOpenAI, city: str) -> str:
    response = await ai.chat.completions.create(
        model=settings.openai_model,
        messages=[{"role": "user", "content": f"Which country is {city} in? One word."}],
        temperature=0,
    )
    return response.choices[0].message.content


async def one_at_a_time() -> list[str]:
    async with async_client() as ai:
        answers = []
        for city in CITIES:
            answers.append(await ask_country(ai, city))  # wait for each before the next
        return answers


async def all_at_once() -> list[str]:
    async with async_client() as ai:
        tasks = []
        for city in CITIES:
            tasks.append(ask_country(ai, city))  # nothing sent yet, just prepared
        return await asyncio.gather(*tasks)  # send all, wait once


started = time.perf_counter()
sequential = run_async(one_at_a_time())
sequential_seconds = time.perf_counter() - started

started = time.perf_counter()
concurrent = run_async(all_at_once())
concurrent_seconds = time.perf_counter() - started

print(f"one at a time: {sequential_seconds:.2f}s | gather: {concurrent_seconds:.2f}s")
print("answers:", concurrent)
assert concurrent == sequential and concurrent_seconds < sequential_seconds

one at a time: 3.36s | gather: 0.72s
answers: ['India.', 'France.', 'Norway.', 'Japan.', 'Peru.']


**What the output shows.** The gather run is close to the time of a single call: the five calls overlapped. Answers are identical and in input order.

**In practice**
- **rate limits** — a burst of 1,000 requests hits the provider's per-minute quota; you get 429s instead of speed. Cap concurrency (next item), or you have traded a slow job for a failed one.
- **partial failure** — with the default, one timeout in 1,000 raises and you lose the other 999 results. `return_exceptions=True`, then handle failures per item and retry only those.
- **memory** — every coroutine holds its request until it finishes; 10,000 is fine, a million is not. Process in chunks of a few hundred.
- **order vs speed** — `gather` preserves input order; to handle results as they arrive, use `asyncio.as_completed`.
- **the client** — create the `AsyncOpenAI` client inside the running loop; one created elsewhere fails with "Event loop is closed".

**Alternatives** — `asyncio.TaskGroup` (Python 3.11+: if one task fails the others are cancelled cleanly) · a job queue with workers (Celery, RQ) when the batch outlives one process

**Terms** — *coroutine*: a prepared async call, not yet running · *in flight*: sent, not yet answered · *`*tasks`*: unpack a list into separate arguments


### concurrency for API calls

> **Problem.** The same indexing job, now with `gather`, fires 2,000 requests in one instant. The provider allows 500 a minute. It answers *429 — too many requests* to 1,500 of them; the SDK retries, the retries also collide, and the job that should have taken four minutes fails after ten.

**Idea.** Two limits on outgoing calls: how many are in flight at once, and how many start per second.

**Use when** always, for any service that calls a provider.  
**Not when** never skip it; only the numbers change per provider and tier.

```mermaid
flowchart LR
    Q[2,000 requests] --> S[semaphore · max 5 in flight] --> R[rate limiter · max 10 / s] --> P[provider · never 429]
```

**How it works.**
1. `asyncio.Semaphore(5)` is a counter with 5 slots. `async with semaphore:` takes a slot, or waits until one is free, and gives it back at the end of the block.
2. `AsyncLimiter(10, time_period=1)` allows 10 starts per second; the 11th waits for the next second.
3. Every request goes through both: `async with semaphore, limiter:` then the call.
4. `gather` still starts all 20 coroutines, but only 5 run and only 10 begin per second — the rest queue inside the semaphore, in memory, costing nothing.
5. The job finishes as fast as the quota allows, with zero 429s.

| | what happens | result |
|:--|:--|:--|
| ✗ no limits | 2,000 sent at once | 429 storm, job fails |
| ✓ semaphore 5 + 10/s | queued and paced | 20 requests in ≈2 s, zero errors |

**Production code and its real output**

In [4]:
# Concurrency control — a semaphore caps requests in flight; aiolimiter caps requests per second.
# Together they keep a burst of work under the provider's rate limits.
import time

from aiolimiter import AsyncLimiter

PROMPTS = []
for n in range(1, 21):
    PROMPTS.append(f"What is {n} squared? Reply with the number only.")
MAX_IN_FLIGHT = 5
REQUESTS_PER_SECOND = 10


async def run_batch() -> list[str]:
    semaphore = asyncio.Semaphore(MAX_IN_FLIGHT)
    limiter = AsyncLimiter(REQUESTS_PER_SECOND, time_period=1)

    async with async_client() as ai:

        async def ask(prompt: str) -> str:
            async with semaphore, limiter:
                response = await ai.chat.completions.create(
                    model=settings.openai_model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0,
                    max_tokens=4,
                )
                return response.choices[0].message.content.strip()

        tasks = []
        for prompt in PROMPTS:
            tasks.append(ask(prompt))
        return await asyncio.gather(*tasks)


started = time.perf_counter()
answers = run_async(run_batch())
seconds = time.perf_counter() - started
print(
    
        f"{len(PROMPTS)} requests in {seconds:.2f}s (max {MAX_IN_FLIGHT} in flight, "
        f"{REQUESTS_PER_SECOND}/s)"
    
)
print("answers:", answers[:6], "...")
assert answers[:3] == ["1", "4", "9"] and seconds >= 1.0

20 requests in 2.67s (max 5 in flight, 10/s)
answers: ['1', '4', '9', '16', '25', '36'] ...


**What the output shows.** 20 requests took about two seconds: the limiter paced them at ~10 per second while the semaphore kept at most 5 open. Answers are correct and in order.

**In practice**
- **per account** — limits are shared across everything using the key: 4 workers × 10 rps = 40 rps. Divide the quota by the number of processes.
- **read the headers** — `x-ratelimit-limit-requests` and `-tokens` on every reply tell you the real budget; configure the limiter from them, not from a guess.
- **tokens, not just requests** — providers also cap tokens per minute; a few long prompts can exhaust that before the request limit — limit both.
- **still back off** — the limiter is a prediction; an actual 429 is the truth. Keep retry-with-backoff on top of it.
- **observe** — expose queue depth and wait time as metrics, or you will not know you are throttling yourself.

**Alternatives** — a gateway (LiteLLM, an API gateway) that enforces limits for all services in one place · the provider's batch API for offline work (half price, no rate pressure)

**Terms** — *semaphore*: a counter that lets N callers through at once · *rate limiter*: allows N starts per second · *429*: the provider's "slow down" reply
